<a href="https://colab.research.google.com/github/putrizaskia358-creator/RegresiBerganda_Batam/blob/main/Regresi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regresi Linear Berganda --Kota Batam

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go


# import Data

In [3]:
# 1. Memuat dataset dari URL
url = "https://raw.githubusercontent.com/Alam-A99/DATAAKSI/main/dataset_fiskal_97_hipoFull.csv"
data_all = pd.read_csv(url)

# 2. Memfilter data
df = data_all[data_all["Kota"] == "Kota Batam"].reset_index(drop=True)

# 3. Ambil hanya variabel yang dipakai
df = df[["Tahun", "DBH_Lainnya", "DAK_Nonfisik", "PE"]]

# 4. Menampilkan hasil
print(df)
print("\nJumlah observasi:", len(df))

    Tahun  DBH_Lainnya  DAK_Nonfisik    PE
0    2014        69.11        177.43  5.52
1    2015        86.96        107.81  7.00
2    2016        92.64        135.86  2.43
3    2017        97.29        142.52  6.14
4    2018       105.68        218.51  5.20
5    2019        73.73        197.15  4.27
6    2020        87.61        170.19  4.04
7    2021       110.23        152.95  5.72
8    2022        65.94        120.59  7.35
9    2023       111.76        249.01  4.98
10   2024       140.09        293.05  6.95

Jumlah observasi: 11


# regresi

In [5]:
# 1. Memuat dataset
url = "https://raw.githubusercontent.com/Alam-A99/DATAAKSI/main/dataset_fiskal_97_hipoFull.csv"
data_all = pd.read_csv(url)

# 2. Memfilter data khusus untuk Kota Batam & membersihkan baris kosong
df = data_all[data_all["Kota"] == "Kota Batam"].reset_index(drop=True)
df = df[["Tahun", "DBH_Lainnya", "DAK_Nonfisik", "PE"]].dropna()

# 3. VARIABEL INDEPENDEN & DEPENDEN
X = df[["DBH_Lainnya", "DAK_Nonfisik"]]
y = df["PE"]

X_model = sm.add_constant(X)

model = sm.OLS(y, X_model)
results = model.fit()

print("\n")
print("=" * 70)
print("HASIL REGRESI LINEAR BERGANDA – KOTA BATAM")
print("=" * 70)
print(results.summary())

# 4. MENGAMBIL KOEFISIEN REGRESI & R²
intercept = results.params["const"]
coef_dbh = results.params["DBH_Lainnya"]
coef_dak = results.params["DAK_Nonfisik"]

r_squared = results.rsquared
adjusted_r_squared = results.rsquared_adj

# Membuat Persamaan Regresi
persamaan = (
    f"Ŷ = {intercept:.3f}"
    f" + ({coef_dbh:.3f}) X₁"
    f" + ({coef_dak:.3f}) X₂"
)

print("\n")
print("=" * 70)
print("                    PERSAMAAN REGRESI")
print("=" * 70)
print(persamaan)
print("X₁ = DBH_Lainnya, X₂ = DAK_Nonfisik, Y = PE (Pertumbuhan Ekonomi)")
print(f"\nR²            = {r_squared:.4f}")
print(f"Adjusted R²   = {adjusted_r_squared:.4f}")
print(f"F-statistic   = {results.fvalue:.3f} (p = {results.f_pvalue:.4f})")

# 5. MENGHITUNG NILAI PREDIKSI & RESIDUAL
df["Prediksi"] = results.predict(X_model)
df["Residual"] = df["PE"] - df["Prediksi"]

print("\n")
print("=" * 70)
print("                  DATA AKTUAL & PREDIKSI")
print("=" * 70)
print(df[["Tahun", "DBH_Lainnya", "DAK_Nonfisik", "PE", "Prediksi", "Residual"]].round(2))





HASIL REGRESI LINEAR BERGANDA – KOTA BATAM
                            OLS Regression Results                            
Dep. Variable:                     PE   R-squared:                       0.024
Model:                            OLS   Adj. R-squared:                 -0.220
Method:                 Least Squares   F-statistic:                   0.09976
Date:                Tue, 13 Oct 2026   Prob (F-statistic):              0.906
Time:                        04:16:32   Log-Likelihood:                -19.182
No. Observations:                  11   AIC:                             44.36
Df Residuals:                       8   BIC:                             45.56
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
con

# menampilkan visualisasi

In [6]:
# ==========================================
# 6. MEMBUAT VISUALISASI 3D INTERAKTIF
# ==========================================
# Membuat grid untuk bidang regresi
x1 = np.linspace(df["DBH_Lainnya"].min(), df["DBH_Lainnya"].max(), 40)
x2 = np.linspace(df["DAK_Nonfisik"].min(), df["DAK_Nonfisik"].max(), 40)
X1, X2 = np.meshgrid(x1, x2)

# Menghitung nilai Y pada bidang regresi
Y = intercept + coef_dbh * X1 + coef_dak * X2

# Membuat figure Plotly 3D
fig = go.Figure()

# Menampilkan Data Aktual (Titik Merah)
fig.add_trace(
    go.Scatter3d(
        x=df["DBH_Lainnya"],
        y=df["DAK_Nonfisik"],
        z=df["PE"],
        mode="markers",
        marker=dict(size=7, color="red", opacity=0.9),
        name="Data Aktual",
        text=[f"Tahun {t}" for t in df["Tahun"]],
        hovertemplate=(
            "<b>%{text}</b><br>"
            "DBH Lainnya: %{x}<br>"
            "DAK Nonfisik: %{y}<br>"
            "Pertumbuhan Ekonomi: %{z}<br>"
            "<extra></extra>"
        ),
    )
)

# Menampilkan Bidang Regresi (Surface)
fig.add_trace(
    go.Surface(
        x=X1, y=X2, z=Y,
        opacity=0.60,
        colorscale="Viridis",
        name="Regression Plane",
        hovertemplate=(
            "DBH Lainnya: %{x:.2f}<br>"
            "DAK Nonfisik: %{y:.2f}<br>"
            "Prediksi PE: %{z:.2f}<br>"
            "<extra></extra>"
        ),
    )
)

# Mengatur tampilan tata letak grafik
fig.update_layout(
    title={
        "text": (
            "REGRESI LINEAR BERGANDA 3D – KOTA BATAM<br>"
            f"<sub>{persamaan} | R² = {r_squared:.3f}</sub>"
        ),
        "x": 0.5,
    },
    scene={
        "xaxis": {"title": "DBH Lainnya (X₁)"},
        "yaxis": {"title": "DAK Nonfisik (X₂)"},
        "zaxis": {"title": "Pertumbuhan Ekonomi (Y)"},
    },
    width=1000,
    height=750,
    margin={"l": 0, "r": 0, "b": 0, "t": 100},
    legend={"x": 0.02, "y": 0.98},
)

# Menampilkan grafik interaktif
fig.show()

# Menyimpan hasil visualisasi ke dalam file HTML
fig.write_html("regresi_3d_kota_batam.html")
print("\n[INFO] Grafik berhasil disimpan sebagai file 'regresi_3d_kota_batam.html'")


[INFO] Grafik berhasil disimpan sebagai file 'regresi_3d_kota_batam.html'
